<a href="https://colab.research.google.com/github/hiten4/Day32PM/blob/main/Day32_PM_PartA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 32 (PM) — Part A: Insurance Fraud Detection using DT & RF


## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score


## 2. Create Synthetic Dataset (3000 records)

In [2]:
np.random.seed(42)

n = 3000

data = pd.DataFrame({
    'claim_amount': np.random.randint(1000, 50000, n),
    'num_previous_claims': np.random.randint(0, 10, n),
    'policy_age': np.random.randint(1, 20, n),
    'customer_age': np.random.randint(18, 80, n),
    'num_dependents': np.random.randint(0, 5, n),
    'fraud_score': np.random.uniform(0, 1, n)
})

# Target (fraud)
data['fraud'] = ((data['fraud_score'] > 0.7) |
                 (data['num_previous_claims'] > 5) |
                 (data['claim_amount'] > 30000)).astype(int)

data.head()

,claim_amount,num_previous_claims,policy_age,customer_age,num_dependents,fraud_score,fraud
0,16795,3,3,79,4,0.480741,0
1,1860,7,2,60,0,0.002963,1
2,39158,6,3,66,4,0.242801,1
3,45732,4,8,68,2,0.323769,1
4,12284,9,16,71,1,0.965994,1


## 3. Train-Test Split

In [3]:
X = data.drop('fraud', axis=1)
y = data['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 4. Decision Tree (max_depth=5)

In [4]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

rules = export_text(dt, feature_names=list(X.columns))
print(rules)


|--- claim_amount <= 29961.00
|   |--- num_previous_claims <= 5.50
|   |   |--- fraud_score <= 0.70
|   |   |   |--- class: 0
|   |   |--- fraud_score >  0.70
|   |   |   |--- class: 1
|   |--- num_previous_claims >  5.50
|   |   |--- class: 1
|--- claim_amount >  29961.00
|   |--- class: 1



## 5. Random Forest (Optimize for Recall)

In [5]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10]
}

rf = RandomForestClassifier(random_state=42)

search = RandomizedSearchCV(
    rf,
    param_dist,
    n_iter=5,
    cv=5,
    scoring='recall',
    random_state=42
)

search.fit(X_train, y_train)
best_rf = search.best_estimator_


## 6. Model Evaluation

In [6]:
def evaluate(model):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Recall": recall_score(y_test, y_pred)
    }

dt_metrics = evaluate(dt)
rf_metrics = evaluate(best_rf)

print("Decision Tree:", dt_metrics)
print("Random Forest:", rf_metrics)


Decision Tree: {'Accuracy': 0.9983333333333333, 'F1': 0.9988331388564761, 'ROC-AUC': np.float64(0.997093023255814), 'Recall': 1.0}
Random Forest: {'Accuracy': 0.9983333333333333, 'F1': 0.9988331388564761, 'ROC-AUC': np.float64(1.0), 'Recall': 1.0}


## 7. Cost-Sensitive Evaluation

In [7]:
def cost_analysis(model):
    y_pred = model.predict(X_test)

    fp = ((y_test == 0) & (y_pred == 1)).sum()
    fn = ((y_test == 1) & (y_pred == 0)).sum()

    cost = (fp * 1) + (fn * 10)
    return cost

print("DT Cost:", cost_analysis(dt))
print("RF Cost:", cost_analysis(best_rf))


DT Cost: 1
RF Cost: 1


## 8. Deployment Recommendation

Random Forest should be used for automated fraud detection because it provides higher recall and better overall performance, which is important since missing fraud cases is very costly.

Decision Tree should be used alongside for interpretability, as it provides clear rules that can be explained to regulators and support human review.